# Train WEPR on labelled answers

The four published detectors each belong to one model, so scoring any other model means
training a detector for it. This notebook is the training half of that, in isolation: it
starts from 50 answers that have **already been generated and already been judged**, and
fits a WEPR detector on them. No GPU, no endpoint, no `vllm` — it runs offline in a few
seconds.

That is the whole point of splitting it this way. Producing the labelled answers is the
expensive part and it needs a model; fitting on them is seconds and needs nothing. Once
you have a file like this one for your own model, the rest of this notebook is what you
do with it.

Where the data comes from, when it is yours:

| Route | What it does |
|---|---|
| {doc}`train_wepr_pipeline` | The same steps against an OpenAI-compatible endpoint, for any QA dataset |
| [`scripts/ecir`](https://github.com/artefactory/artefactual/tree/main/scripts/ecir) | The paper's batch procedure — `vllm run-batch` and an LLM judge |

> **The shipped sample is synthetic.** The questions and gold answers are real trivia, but
> no model produced these answers and no judge graded them — a stand-in answers correctly
> or, 40% of the time, with a fixed wrong answer, and its per-token log-probabilities are
> drawn wider when it is wrong. The training path below is therefore real, and the scores
> it reports describe the simulation rather than any model. The file's own `note` field
> says the same thing. Swap in a real run and every number becomes meaningful.

## 1. What a labelled example looks like

Three things per example, and only the third is unusual:

- the **generated answer**, which the model produced;
- the **label**, `1` for a hallucination — here from `hallucination`, in a real run from
  an LLM judge's verdict, negated (`judgment: true` means the answer was *correct*);
- the **response**, carrying `logprobs.content` with `top_logprobs` per token. This is
  what the detector reads. An answer without it cannot be scored at all.

In [1]:
import json
from pathlib import Path

sample = json.loads(Path("wepr_training_sample.json").read_text(encoding="utf-8"))

# Ranks per token. Part of the feature definition, not a batch size: WEPR fits one
# coefficient per rank, so the detector must later be loaded at the value it was fit at.
K = sample["k"]

print(f"{sample['n_examples']} examples, {sample['n_hallucinations']} hallucinations, k={K}")
print()
print(sample["note"])

50 examples, 19 hallucinations, k=15

SYNTHETIC DATA. The questions and their gold answers are real trivia; everything else is simulated. No language model produced these answers and no judge graded them: for each question a stand-in model answers correctly or, 40% of the time, with a fixed wrong answer, and the per-token top-15 log-probabilities are drawn from a half-normal whose spread is wider when the answer is wrong -- overlapping, so some wrong answers are confident and some right ones hesitant. That is the hesitation EPR and WEPR measure, so the training path exercised by the notebook is the real one -- but the scores it reports describe this simulation, not any model's behaviour: the classes are separable here because they were built that way, and a quarter of 50 is 13 examples, which cannot measure anything. For numbers about a real model, generate against it: see the train_wepr_pipeline notebook, or scripts/ecir for the paper's batch procedure.


In [2]:
example = sample["examples"][0]

print("question:      ", example["question"])
print("gold answer:   ", example["short_answer"])
print("model said:    ", example["generated_answer"])
print("hallucination: ", example["hallucination"])

first_token = example["response"]["choices"][0]["logprobs"]["content"][0]
print(f"\nfirst token {first_token['token']!r} carries {len(first_token['top_logprobs'])} ranks:")
for rank in first_token["top_logprobs"][:4]:
    print(f"    {rank['token']:>8}  {rank['logprob']:.4f}")
print("     ...")

question:       Who wrote the novel 'Things Fall Apart'?
gold answer:    Chinua Achebe
model said:     Wole Soyinka
hallucination:  1

first token 'Wole' carries 15 ranks:
        Wole  -0.0177
      «alt1»  -0.0181
      «alt2»  -0.0555
      «alt3»  -0.1226
     ...


## 2. Split the examples

`responses` are passed to the detector exactly as they came back from the provider — the
pipeline's first step parses them, so there is no feature extraction to write.

The split is stratified, so the hallucination rate is the same on both sides, and it
happens before the fit: the scores in step 4 are then about answers the detector has never
seen. With 50 examples a quarter is 13, which is small — wide enough intervals that you
should read them as a smoke test of the procedure, not as a measurement. More data is the
only fix, and it is the reason a real run wants a few hundred questions.

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split

responses = [example["response"] for example in sample["examples"]]
y = np.array([example["hallucination"] for example in sample["examples"]])

# Both classes are needed. All-correct means the questions were too easy for the model;
# all-wrong usually means it is not answering in the expected short form.
assert 0 < y.sum() < len(y), f"labels are single-class ({y.sum()}/{len(y)})"

x_train, x_test, y_train, y_test = train_test_split(responses, y, test_size=0.25, stratify=y, random_state=42)
print(f"{len(y_train)} to fit on, {len(y_test)} held out ({y_test.sum()} hallucinations)")

37 to fit on, 13 held out (5 hallucinations)


## 3. Fit

`trainable=True` returns an unfitted pipeline — parser, entropy reduction, logistic
regression. It is explicit by design: calling `wepr()` with neither weights nor
`trainable=True` raises, rather than handing back a detector that would emit probabilities
no trained weights support.

WEPR keeps one coefficient per rank, in both directions, so a `k=15` fit has 30 of them.
`epr` is the same call with one feature instead, pooling every rank into a single number.

In [4]:
from artefactual.scoring import wepr

detector = wepr(k=K, trainable=True).fit(x_train, y_train)

classifier = detector.named_steps["classifier"]
print(f"{classifier.coef_.shape[1]} coefficients (2k = {2 * K}), intercept {classifier.intercept_[0]:.3f}")
detector

30 coefficients (2k = 30), intercept 190.707


,steps,"[('parser', ...), ('entropy', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
Name,Type,Value
classes_,"ndarray[int64](2,)","[0,1]"
,k,15
,reduction,'wepr'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",inf
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'


## 4. Score it on the held-out examples

Two different questions, and both are worth reading. **ROC-AUC** scores the *ranking* —
whether hallucinations get higher scores than grounded answers — which is what governs
triage by score, and what the paper reports. The **classification report** scores the
decisions at the 0.5 threshold; recall on the `hallucination` row is the fraction of
hallucinations actually flagged.

A detector can rank well and still decide poorly at 0.5. Only the AUC carries over to
another threshold, so pick the threshold from these scores rather than assuming 0.5 —
[the scoring guide](https://artefactory.github.io/artefactual/guide/scoring.html) covers
how.

In [5]:
from sklearn.metrics import classification_report, roc_auc_score

scores = detector.predict_proba(x_test)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, scores):.3f}   (on {len(y_test)} held-out examples)")
print(classification_report(y_test, scores >= 0.5, target_names=["grounded", "hallucination"], zero_division=0))

ROC-AUC: 0.875   (on 13 held-out examples)
               precision    recall  f1-score   support

     grounded       1.00      0.50      0.67         8
hallucination       0.56      1.00      0.71         5

     accuracy                           0.69        13
    macro avg       0.78      0.75      0.69        13
 weighted avg       0.83      0.69      0.68        13



## 5. Save the weights, and use them

The file is the same `.skops` format the published detectors ship in, so it loads through
the same call — a repository id, a path, either one. `k` has to be the value the weights
were fitted at; another value raises rather than mis-shaping the score.

From here it is an ordinary detector: `predict_proba` for one score per response,
`predict_token_proba` for where in the answer the model started drifting.

Expect the probabilities below to sit hard against 0 and 1. The published detectors fit an
unregularised logistic regression (`C=inf`), and on 37 examples that drives the
coefficients large enough to saturate the sigmoid — so the *ordering* of these scores
carries the information and their distance from 0.5 does not. It is another reason the AUC
above is the number to read, and another reason to train on more than 50 examples.

In [6]:
path = detector.save_estimator("wepr-demo.skops")
reloaded = wepr(path, k=K)

for example in sample["examples"][:4]:
    probability = reloaded.predict_proba(example["response"])[0, 1]
    marker = "hallucination" if example["hallucination"] else "grounded    "
    print(f"[{marker}] P={probability:.3f}  {example['generated_answer']!r}  (gold: {example['short_answer']!r})")

[hallucination] P=1.000  'Wole Soyinka'  (gold: 'Chinua Achebe')
[grounded    ] P=0.000  'Ulaanbaatar'  (gold: 'Ulaanbaatar')
[grounded    ] P=0.000  'Gold'  (gold: 'Gold')
[grounded    ] P=0.000  '1989'  (gold: '1989')


In [7]:
token_scores = reloaded.predict_token_proba(sample["examples"][0]["response"])[0, :, 0]
tokens = [t["token"] for t in sample["examples"][0]["response"]["choices"][0]["logprobs"]["content"]]

for token, score in zip(tokens, token_scores):
    print(f"    {token:>15}  {score:.3f}")

               Wole  1.000
            Soyinka  0.000


## Where to go next

- **Bring your own labelled file.** Anything with a response per answer and a 0/1 label
  works; only step 1's field names change. The response has to carry `top_logprobs` at
  `k` or wider — wider is fine, surplus ranks are dropped, and narrower is refused rather
  than zero-filled.
- **Generate and judge it yourself.** {doc}`train_wepr_pipeline` runs the same steps as
  this notebook's data came from, against an endpoint, for any QA dataset with answers.
- **Reproduce the paper.** [`scripts/ecir`](https://github.com/artefactory/artefactual/tree/main/scripts/ecir)
  runs both LLM stages as `vllm run-batch` jobs, which is the practical way to label
  thousands of questions.
- **More examples.** 50 is enough to see the mechanics, not to measure anything. A few
  hundred is a workable start.